# Exercise 4.4: Accessibility revised

From *Programming in High Energy Particle Physics*, Chapter 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch04/ex04_4_solution.ipynb)

Exercise 4.4 Accessibility revised Create a data/MC comparison plot using a colorblind-safe palette, either the Okabe–Ito palette or the Petroff palette recommended by CMS ( Table 4.1 ). Test it by converting the figure to grayscale with the luminance formula \(Y = 0.299R + 0.587G + 0.114B\), and by simulating deuteranopia and protanopia with colorspacious or DaltonLens; verify that all histogram stacks remain distinguishable. Include different line styles (solid, dashed, dotted) as secondary visual cues.

<details><summary>Hint</summary>

Save the figure as PNG, load it with matplotlib.pyplot.imread , and apply the luminance weights to the RGB channels to get a grayscale image; display it with cmap="gray" . For the CVD check, pass the RGB array to colorspacious.cspace_convert as in Section 4.1.2 . Neighboring stack components should differ in lightness, not only in hue.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy matplotlib colorspacious")

### Plot and inspect color vision variants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from colorspacious import cspace_convert
rng=np.random.default_rng(42)
edges=np.linspace(60,120,25);centers=(edges[:-1]+edges[1:])/2
width=edges[1]-edges[0]
z=1000*np.exp(-0.5*((centers-91.1876)/3.2)**2);z=z/z.sum()*1200
bg1=9*np.exp(-(centers-60)/35);bg2=3*np.exp(-(centers-60)/18)
components=[bg1,bg2,z]
colors=['#000000','#F0E442','#56B4E9']  # Okabe–Ito palette
labels=['Other background','Continuum','Z → ℓℓ']
data=rng.poisson(sum(components));mc=sum(components)
fig,(ax,rax)=plt.subplots(2,1,figsize=(8,7),sharex=True,
    gridspec_kw={'height_ratios':[3,1],'hspace':0.05})
bottom=np.zeros_like(centers)
for comp,color,label,style in zip(components,colors,labels,['-','--',':']):
    ax.bar(centers,comp,bottom=bottom,width=width,color=color,label=label,
           edgecolor='white',linewidth=0.5)
    bottom=bottom+comp
    ax.step(edges,np.r_[bottom,bottom[-1]],where='post',color='black',linestyle=style,lw=1.2)
ax.errorbar(centers,data,yerr=np.sqrt(data),fmt='ko',markersize=3,label='Pseudo-data')
ax.set_ylabel(f'Events / {width:.1f} GeV')
ax.text(0.03,0.96,'CMS-style  Toy simulation',transform=ax.transAxes,va='top')
ax.legend(frameon=False,fontsize=9)
ratio=np.divide(data,mc,out=np.ones_like(mc),where=mc>0)
rax.errorbar(centers,ratio,yerr=np.sqrt(data)/mc,fmt='ko',markersize=3)
rax.axhline(1,color='black');rax.set_xlabel(r'$m_{\ell\ell}$ [GeV]')
rax.set_ylabel('Data/MC');rax.set_ylim(0.4,1.6)
fig.savefig('accessible_datamc.png',dpi=150,bbox_inches='tight');plt.show()
rgb=np.array([to_rgb(c) for c in colors])
luminance=rgb @ np.array([0.299,0.587,0.114])
print('Component luminances:',luminance)
assert min(abs(np.diff(luminance)))>0.1
fig_rgb=plt.imread('accessible_datamc.png')[...,:3]
gray=fig_rgb @ np.array([0.299,0.587,0.114])
fig2,ax2=plt.subplots(figsize=(8,5));ax2.imshow(gray,cmap='gray')
ax2.axis('off');ax2.set_title('Grayscale inspection');plt.show()
for cvd in ('deuteranomaly','protanomaly'):
    converted=np.clip(cspace_convert(rgb,'sRGB1',
      {'name':'sRGB1+CVD','cvd_type':cvd,'severity':100}),0,1)
    distances=np.linalg.norm(np.diff(converted,axis=0),axis=1)
    print(cvd,'adjacent RGB distances:',distances)
    assert np.min(distances)>0.15
    strip=np.tile(converted[:,None,:],(1,60,1))
    plt.figure(figsize=(4,1.5));plt.imshow(strip,aspect='auto');plt.title(cvd)
    plt.yticks(range(3),labels);plt.xticks([]);plt.show()

In [ ]:
assert np.isfinite(ratio).all()

### Interpretation

The palette has distinct neighboring luminances and solid, dashed and dotted boundaries. Numerical checks and rendered views complement visual inspection; they cannot guarantee readability for every viewer or display.